In [23]:
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

## Load Dataset

In [24]:
df = sns.load_dataset("titanic")

print("First rows:")
print(df.head())

First rows:
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  


In [25]:
print("\nDataset shape:")
print(df.shape)


Dataset shape:
(891, 15)


In [26]:
print("\nMissing values:")
print(df.isnull().sum())


Missing values:
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


## Select useful columns

In [33]:
df = df[
    [
        "survived",
        "pclass",
        "sex",
        "age",
        "sibsp",
        "parch",
        "fare",
        "embarked",
        "alone"
    ]
].copy()

## Feature Engineering

In [34]:
df["family_size"] = df["sibsp"] + df["parch"] + 1
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,alone,family_size
0,0,3,male,22.0,1,0,7.2500,S,False,2
1,1,1,female,38.0,1,0,71.2833,C,False,2
2,1,3,female,26.0,0,0,7.9250,S,True,1
3,1,1,female,35.0,1,0,53.1000,S,False,2
4,0,3,male,35.0,0,0,8.0500,S,True,1


## Extract Labels and Features

In [35]:
X = df.drop("survived", axis=1)
y = df["survived"]

## Preprocessing

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "bool", "category", "str"]).columns
X[categorical_features] = X[categorical_features].astype(str)

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")), # Replace NAN-Values with median
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")), # Replace NAN-Values with mode
        ("encoder", OneHotEncoder(handle_unknown="ignore")) # male --> [0, 1, 0] female [1, 0, 0] child [0, 0, 1]
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## Split the Data

In [40]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [44]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

X_train_df = pd.DataFrame(
    X_train_prepared,
    columns=feature_names,
    index=X_train.index
)

X_test_df = pd.DataFrame(
    X_test_prepared,
    columns=feature_names,
    index=X_test.index
)

print(X_train_df.head())

     num__pclass  num__age  num__sibsp  num__parch  num__fare  \
692     0.829568 -0.081135   -0.465084   -0.466183   0.513812   
481    -0.370945 -0.081135   -0.465084   -0.466183  -0.662563   
527    -1.571457 -0.081135   -0.465084   -0.466183   3.955399   
855     0.829568 -0.887827   -0.465084    0.727782  -0.467874   
801    -0.370945  0.110934    0.478335    0.727782  -0.115977   

     num__family_size  cat__sex_female  cat__sex_male  cat__embarked_C  \
692         -0.556339              0.0            1.0              0.0   
481         -0.556339              0.0            1.0              0.0   
527         -0.556339              0.0            1.0              0.0   
855          0.073412              1.0            0.0              0.0   
801          0.703162              1.0            0.0              0.0   

     cat__embarked_Q  cat__embarked_S  cat__alone_False  cat__alone_True  
692              0.0              1.0               0.0              1.0  
481           